# 0liviA — Kaggle T4×2 offline benchmark

**Uso exclusivo:** benchmark privado, manual y acotado de modelos ML con datos sintéticos.

**No es un endpoint:** esta notebook no inicia un servidor, túnel, proxy, heartbeat ni conexión con 0liviA / GitHub Actions. No cargar chats, secretos, credenciales ni datos privados.

Antes de activar GPU: revisar AUP/Terms y cuota real de la cuenta. Hardware y cuota pueden no estar disponibles en cada sesión.

In [ ]:
import json, os, platform, subprocess, sys, time, hashlib
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Notebook purpose: offline ML benchmark only")
print("No HTTP server / tunnel / Core connection is part of this notebook.")

In [ ]:
LLAMA_CPP_SHA = "2cbca389f515df455412dafa07e2b1359cc72a6d"
CUDA_ARCH = "75"
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, *, check=True, capture=False):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )

gpu = run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture=True)
print(gpu.stdout)
import torch
assert torch.cuda.is_available(), "CUDA is unavailable; stop before building/loading."
assert torch.cuda.device_count() == 2, f"Expected 2 GPUs, got {torch.cuda.device_count()}"
names = [torch.cuda.get_device_name(i) for i in range(2)]
caps = [torch.cuda.get_device_capability(i) for i in range(2)]
assert all("T4" in n for n in names), f"Expected Tesla T4 x2, got {names}"
assert all(c == (7, 5) for c in caps), f"Expected SM 7.5, got {caps}"
print({"gpu_names": names, "compute_capability": caps, "cpu_count": os.cpu_count()})

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
manifests = list(INPUT_ROOT.rglob("manifest.json"))
assert len(manifests) == 1, (
    f"Expected exactly one attached model manifest under {INPUT_ROOT}; found {len(manifests)}. "
    "Attach one private dataset containing one GGUF and one manifest.json."
)
MANIFEST_PATH = manifests[0]
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
required = ["model_id", "source_url", "license", "quantization", "file", "sha256"]
missing = [k for k in required if not manifest.get(k)]
assert not missing, f"Manifest missing required fields: {missing}"
MODEL_PATH = MANIFEST_PATH.parent / manifest["file"]
assert MODEL_PATH.is_file(), f"GGUF not found: {MODEL_PATH}"
print("Model:", manifest["model_id"])
print("Source:", manifest["source_url"])
print("License:", manifest["license"])
print("Quant:", manifest["quantization"])
print("File size GiB:", round(MODEL_PATH.stat().st_size / 1024**3, 2))

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

actual_sha = sha256_file(MODEL_PATH)
expected_sha = str(manifest["sha256"]).lower().strip()
assert len(expected_sha) == 64 and all(c in "0123456789abcdef" for c in expected_sha), "Manifest SHA-256 is not a real 64-hex digest."
assert actual_sha == expected_sha, f"SHA-256 mismatch: expected {expected_sha}, got {actual_sha}"
print("SHA256 verified:", actual_sha)

In [ ]:
LLAMA_DIR = Path("/kaggle/working/llama.cpp")
if not LLAMA_DIR.exists():
    run(["git", "clone", "--filter=blob:none", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_DIR)])
run(["git", "-C", LLAMA_DIR, "fetch", "--depth=1", "origin", LLAMA_CPP_SHA])
run(["git", "-C", LLAMA_DIR, "checkout", "--detach", LLAMA_CPP_SHA])
observed_sha = run(["git", "-C", LLAMA_DIR, "rev-parse", "HEAD"], capture=True).stdout.strip()
assert observed_sha == LLAMA_CPP_SHA, (observed_sha, LLAMA_CPP_SHA)

run(["cmake", "-S", LLAMA_DIR, "-B", LLAMA_DIR / "build",
     "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
     "-DCMAKE_BUILD_TYPE=Release"])
run(["cmake", "--build", LLAMA_DIR / "build", "--target", "llama-cli", "llama-bench", "-j2"])
LLAMA_BENCH = LLAMA_DIR / "build" / "bin" / "llama-bench"
LLAMA_CLI = LLAMA_DIR / "build" / "bin" / "llama-cli"
assert LLAMA_BENCH.is_file() and LLAMA_CLI.is_file()
print("llama.cpp pinned:", observed_sha)

In [ ]:
# Planning estimates only; verify with observed nvidia-smi during every run.
MODEL_KV_KIB_PER_TOKEN = {
    "Qwen3.6-35B-A3B": 20,
    "Qwen3-Coder-30B-A3B-Instruct": 96,
    "Qwen3.8-27B": 64,
}
model_id = manifest["model_id"]
kv_kib = next((v for k, v in MODEL_KV_KIB_PER_TOKEN.items() if k in model_id), None)
assert kv_kib is not None, f"Add a reviewed KV estimate for {model_id}; do not guess."

weights_gib = MODEL_PATH.stat().st_size / 1024**3
contexts = [8192, 16384, 32768]
for ctx in contexts:
    kv_total_gib = kv_kib * ctx / 1024**2
    # Layer split is not guaranteed to be exactly 50/50; use a 55% worst-side estimate.
    worst_gpu_gib = weights_gib * 0.55 + kv_total_gib * 0.55 + 1.5
    print({"context": ctx, "estimated_worst_gpu_gib": round(worst_gpu_gib, 2),
           "kv_total_gib": round(kv_total_gib, 2), "target_max_gib": 13.5})
    if worst_gpu_gib > 13.5:
        print("WARNING: estimate exceeds target; reduce context or select a smaller quant before loading.")


In [ ]:
bench_cmd = [
    LLAMA_BENCH, "-m", MODEL_PATH, "-ngl", "99",
    "-sm", "layer", "-ts", "1,1",
    "-p", "512,2048,4096", "-n", "128", "-r", "3", "-o", "json"
]
bench = run(bench_cmd, check=False, capture=True)
(OUTPUT_DIR / "llama-bench.jsonl").write_text(bench.stdout or "", encoding="utf-8")
print("llama-bench exit:", bench.returncode)
print((bench.stdout or "")[-5000:])
assert bench.returncode == 0, "llama-bench failed; inspect captured output."

In [ ]:
context_results = []
synthetic_prompt = (
    "Implement ten independent Python functions: add(a,b); clamp(x,low,high); "
    "median(values) (ValueError on empty); slugify(text) (lowercase, spaces to hyphens, remove punctuation); "
    "unique_preserving_order(items); is_palindrome(text) (case-insensitive, ignore spaces); "
    "parse_positive_int(value) (ValueError unless positive integer string); "
    "safe_divide(a,b) (ZeroDivisionError when b=0); chunked(items,size) (ValueError for size<=0); "
    "count_words(text) (split on whitespace). Return valid Python code only, no markdown."
)
for ctx in [8192, 16384, 32768]:
    cmd = [LLAMA_CLI, "-m", MODEL_PATH, "-ngl", "99", "-sm", "layer", "-ts", "1,1",
           "-c", str(ctx), "-n", "768", "--temp", "0", "--seed", "42",
           "--no-display-prompt", "-p", synthetic_prompt]
    start = time.perf_counter()
    result = run(cmd, check=False, capture=True)
    elapsed = time.perf_counter() - start
    output_path = OUTPUT_DIR / f"synthetic_ctx_{ctx}.txt"
    output_path.write_text(result.stdout or "", encoding="utf-8")
    context_results.append({"context": ctx, "exit_code": result.returncode,
                            "wall_seconds": round(elapsed, 3), "output_file": str(output_path)})
    print(context_results[-1])
    if result.returncode != 0:
        break
print(context_results)

In [ ]:
# Ten small, independent synthetic coding tasks; no 0liviA source or user data.
candidate = OUTPUT_DIR / "synthetic_ctx_32768.txt"
if not candidate.exists() or not candidate.read_text(encoding="utf-8").strip():
    candidate = next((OUTPUT_DIR / f"synthetic_ctx_{c}.txt" for c in [16384, 8192]
                      if (OUTPUT_DIR / f"synthetic_ctx_{c}.txt").exists()), None)
assert candidate is not None, "No synthetic generation output exists."
raw = candidate.read_text(encoding="utf-8")
lines = raw.strip().splitlines()
if lines and lines[0].lstrip().startswith("```"):
    lines = lines[1:]
if lines and lines[-1].strip() == "```":
    lines = lines[:-1]
generated = "\n".join(lines).strip()
generated_path = OUTPUT_DIR / "generated_solution.py"
generated_path.write_text(generated + "\n", encoding="utf-8")
print("Generated fixture:", generated_path)
print(generated[:6000])

In [ ]:
import ast, unittest

source = generated_path.read_text(encoding="utf-8")
namespace = {}
syntax_error = None
try:
    ast.parse(source)
    exec(compile(source, str(generated_path), "exec"), namespace)
except Exception as exc:
    syntax_error = repr(exc)

cases = [
    ("add", lambda: namespace["add"](2, 3) == 5),
    ("clamp", lambda: namespace["clamp"](9, 0, 5) == 5),
    ("median", lambda: namespace["median"]([1, 9, 3]) == 3),
    ("slugify", lambda: namespace["slugify"]("Hola, Mundo!") == "hola-mundo"),
    ("unique_preserving_order", lambda: namespace["unique_preserving_order"]([2, 1, 2, 3]) == [2, 1, 3]),
    ("is_palindrome", lambda: namespace["is_palindrome"]("Never odd or even") is True),
    ("parse_positive_int", lambda: namespace["parse_positive_int"]("12") == 12),
    ("safe_divide", lambda: namespace["safe_divide"](9, 3) == 3),
    ("chunked", lambda: namespace["chunked"]([1, 2, 3], 2) == [[1, 2], [3]]),
    ("count_words", lambda: namespace["count_words"]("one  two\\nthree") == 3),
]
task_results = []
for name, check in cases:
    try:
        assert syntax_error is None, syntax_error
        passed = bool(check())
        detail = "PASS" if passed else "assertion returned false"
    except Exception as exc:
        passed, detail = False, repr(exc)
    task_results.append({"task": name, "passed": passed, "detail": detail})
    print(task_results[-1])

passed_count = sum(item["passed"] for item in task_results)
print(f"SYNTHETIC_PASS_AT_1={passed_count}/10")


In [ ]:
# Export sanitized evidence. Do not add user prompts, credentials, cookies or private files.
gpu_snapshot = run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
                    "--format=csv,noheader"], check=False, capture=True)
metrics = {
    "scope": "offline_ml_benchmark_only",
    "model": manifest,
    "model_sha256_observed": actual_sha,
    "llama_cpp_sha": observed_sha,
    "gpu_names": names,
    "compute_capability": caps,
    "gpu_memory_snapshot": gpu_snapshot.stdout,
    "context_results": context_results,
    "llama_bench_exit_code": bench.returncode,
    "synthetic_task_results": task_results,
    "synthetic_pass_at_1": passed_count,
    "policy": {"no_server": True, "no_tunnel": True, "no_core_connection": True},
}
metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print("Export:", metrics_path, "bytes:", metrics_path.stat().st_size)
print("Download metrics.json, then stop the Kaggle interactive session manually.")
